In [3]:
import pandas as pd
import numpy as np
from datetime import datetime
import re

# 1. 读取原始供应商表
file_path = r"C:\Users\lenovo\Desktop\供应商名录_正常版.xlsx"
df = pd.read_excel(file_path, sheet_name=0, dtype={"统一社会信用代码": str})
# 信用代码强制文本，避免18位长数字丢失前缀0

# 查看原始信息
print("原始数据行数：", len(df))
print("列名：", df.columns.tolist())

原始数据行数： 177
列名： ['序号', '系统匹配企业名称', '登记状态', '法定代表人', '联系电话', '注册资本', '成立日期', '统一社会信用代码', '企业地址', '企业（机构）类型', '行业', '企业简介', '经营范围', '营业期限', '企业规模', '资质证书', '是否录入供应商名录', '是否失信', '公司经营']


In [4]:
# 自增主键ID（唯一行标识，删除行不会错乱）
df["主键ID"] = range(1, len(df)+1)

# 软删除标记：正常/作废，不物理删除数据
df["数据状态"] = "正常"

# 数据更新追溯字段
df["最后更新时间"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
df["操作人"] = "管理员"

# 把主键ID放第一列，方便检索
cols = ["主键ID"] + df.columns[:-4].tolist() + ["数据状态", "最后更新时间", "操作人"]
df = df[cols]

In [5]:
def clean_text(s):
    """清除空格、换行、特殊符号，空值统一替换"""
    if pd.isna(s):
        return "未公示"
    s = str(s).strip().replace("\n", "").replace("\r", "")
    # 去除多余特殊符号
    s = re.sub(r"[★#&*@]", "", s)
    if s == "" or s.lower() in ["nan", "none"]:
        return "未公示"
    return s

def split_multi_val(s):
    """多值字段统一用英文分号;分隔（资质、经营范围）"""
    s = clean_text(s)
    # 中文顿号、逗号统一替换为;
    s = s.replace("、", ";").replace("，", ";").replace(",", ";")
    # 去除连续分隔符
    s = re.sub(r";+", ";", s)
    return s.strip(";")

In [6]:
# ========== 企业名称、法人、地址文本清洗 ==========
df["系统匹配企业名称"] = df["系统匹配企业名称"].apply(clean_text)
df["法定代表人"] = df["法定代表人"].apply(clean_text)
df["企业地址"] = df["企业地址"].apply(clean_text)
df["企业简介"] = df["企业简介"].apply(clean_text)

# ========== 枚举字段统一标准化（固定值域，筛选查询不乱） ==========
# 登记状态映射
status_map = {
    "正常": "存续", "在业": "存续", "已注销": "注销", "吊销未注销": "吊销",
    "停业": "停业", "清算中": "清算", "迁出": "迁出"
}
df["登记状态"] = df["登记状态"].apply(lambda x: status_map.get(clean_text(x), clean_text(x)))

# 是否录入供应商名录、是否失信 统一为 是/否
def bool_flag(x):
    x = clean_text(x)
    if x in ["是", "已录入", "有", "失信", "列入"]:
        return "是"
    else:
        return "否"
df["是否录入供应商名录"] = df["是否录入供应商名录"].apply(bool_flag)
df["是否失信"] = df["是否失信"].apply(bool_flag)

# 企业规模统一枚举
scale_map = {"小型企业": "小型", "微型企业": "微型", "中型企业": "中型", "大型企业": "大型"}
df["企业规模"] = df["企业规模"].apply(lambda x: scale_map.get(clean_text(x), clean_text(x)))

# 公司经营状态统一
operate_map = {"异常": "经营异常", "正常营业": "正常经营", "倒闭": "破产"}
df["公司经营"] = df["公司经营"].apply(lambda x: operate_map.get(clean_text(x), clean_text(x)))

# ========== 联系电话规整，只保留数字，多号码分号分隔 ==========
def clean_phone(x):
    s = clean_text(x)
    if s == "未公示":
        return "无联系方式"
    # 提取所有数字
    nums = re.findall(r"\d+", s)
    if not nums:
        return "无联系方式"
    return ";".join(nums)
df["联系电话"] = df["联系电话"].apply(clean_phone)

# ========== 注册资本统一数值化（新增数字列用于排序筛选） ==========
def reg_cap_to_num(x):
    s = clean_text(x)
    match = re.search(r"(\d+\.?\d*)", s)
    if match:
        return float(match.group(1))
    return 0.0
df["注册资本文本"] = df["注册资本"].apply(clean_text)
df["注册资本万元"] = df["注册资本"].apply(reg_cap_to_num)

# ========== 日期统一格式 yyyy-MM-dd ==========
def standard_date(x):
    s = clean_text(x)
    if s in ["未公示", "无"]:
        return "未知"
    try:
        dt = pd.to_datetime(s)
        return dt.strftime("%Y-%m-%d")
    except:
        return "未知"
df["成立日期"] = df["成立日期"].apply(standard_date)

# 营业期限统一格式
def clean_term(x):
    s = clean_text(x)
    if "长期" in s or s == "未公示":
        return "2099-12-31"
    return s
df["营业期限"] = df["营业期限"].apply(clean_term)

# ========== 多值字段：经营范围、资质证书统一分隔符 ==========
df["经营范围"] = df["经营范围"].apply(split_multi_val)
df["资质证书"] = df["资质证书"].apply(split_multi_val)

# 统一社会信用代码校验（18位标记异常）
def check_credit_code(x):
    s = clean_text(x)
    if len(s) == 18 and s.isalnum():
        return s
    return f"{s}(代码异常)"
df["统一社会信用代码"] = df["统一社会信用代码"].apply(check_credit_code)

In [7]:
# 标记重复企业
df["重复标记"] = df.duplicated(subset=["统一社会信用代码"], keep=False).map({True: "重复", False: "唯一"})

# 去重，保留第一条数据
df_clean = df.drop_duplicates(subset=["统一社会信用代码"], keep="first")
print(f"去重前：{len(df)} 行，去重后：{len(df_clean)} 行")

去重前：177 行，去重后：173 行


In [8]:
# 附表1：资质明细表（一条资质一行）
qual_list = []
for _, row in df_clean.iterrows():
    credit = row["统一社会信用代码"]
    quals = row["资质证书"].split(";") if row["资质证书"] != "未公示" else []
    for q in quals:
        qual_list.append({"统一社会信用代码": credit, "资质内容": q})
df_qual = pd.DataFrame(qual_list)

# 附表2：经营范围明细表
biz_list = []
for _, row in df_clean.iterrows():
    credit = row["统一社会信用代码"]
    bizs = row["经营范围"].split(";") if row["经营范围"] != "未公示" else []
    for b in bizs:
        biz_list.append({"统一社会信用代码": credit, "经营项目": b})
df_biz = pd.DataFrame(biz_list)

In [9]:
with pd.ExcelWriter("供应商名录_正常版.xlsx", engine="openpyxl") as writer:
    df_clean.to_excel(writer, sheet_name="企业主表", index=False)
    df_qual.to_excel(writer, sheet_name="资质附表", index=False)
    df_biz.to_excel(writer, sheet_name="经营范围附表", index=False)

print("数据规整完成，已输出 供应商名录_正常版.xlsx")

数据规整完成，已输出 供应商名录_正常版.xlsx


In [14]:
# 读取规整后文件
df_main = pd.read_excel("供应商名录_正常版.xlsx", sheet_name="企业主表")

# 1. 精准查询：统一社会信用代码
target = df_main[df_main["统一社会信用代码"] == "91441300XXXXXXXXXX"]

# 2. 条件查询：失信+已录入供应商名录
bad_supp = df_main[(df_main["是否失信"] == "是") & (df_main["是否录入供应商名录"] == "是")]

# 3. 模糊查询企业名称
search = df_main[df_main["系统匹配企业名称"].str.contains("建筑", na=False)]

In [15]:
new_row = {
    "主键ID": df_main["主键ID"].max() + 1,
    "系统匹配企业名称": "杭州数梦工场科技有限公司",
    "统一社会信用代码": "91330106328122968A",
    "登记状态": "存续",
    "法定代表人": "肖艳梅",
    "联系电话": "13800138000",
    "注册资本文本": "19771.658169万元",
    "注册资本万元": 19771.658169,
    "成立日期": "2015-03-16",
    "企业地址": "浙江省杭州市西湖区转塘科技经济区块16号（中大银座）9幢1楼",
    "企业（机构）类型": "有限责任公司",
    "行业": "信息传输、软件和信息技术服务业",
    "企业简介": "杭州数梦工场科技有限公司成立于2015年3月，董事长为黄志宏。公司专注于数据资源体系基础设施的研发与服务，是相关领域的技术供应商。杭州数梦工场科技有限公司入选2024年国家级独角兽企业、2025年国家级专利优秀奖和2025年独角兽企业。杭州市重视数字技术企业在产业数字化领域的发展，为杭州数梦工场科技有限公司提供了良好的政策支持和发展环境。上城高新区聚焦数字能源、智能物联等赛道，通过靶向招引与全周期服务构建特色现代产业矩阵，为杭州数梦工场科技有限公司的技术创新和市场拓展创造了有利条件。",
    "经营范围": "云计算软硬件、大数据的技术开发、技术服务、技术咨询；技术进出口（国家法律法规禁止经营的项目除外，国家法律、法规限制经营的项目取得许可证后方可经营）；销售：本企业自行开发的软硬件产品；批发：计算机软硬件及辅助设备、通信产品。",
    "营业期限": "2015-03-16 至 无固定期限",
    "企业规模": "小型",
    "资质证书": "建筑三级;安全生产许可",
    "是否录入供应商名录": "否",
    "是否失信": "是",
    "公司经营": "异常经营",
    "数据状态": "异常",
    "最后更新时间": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "操作人": "管理员"
}
# 追加到主表
df_main = pd.concat([df_main, pd.DataFrame([new_row])], ignore_index=True)

In [12]:
# 根据信用代码定位行，修改法人和电话
mask = df_main["统一社会信用代码"] == "91441300XXXXXXXXXX"
df_main.loc[mask, "法定代表人"] = "李四"
df_main.loc[mask, "联系电话"] = "13900139000"
df_main.loc[mask, "最后更新时间"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [13]:
mask = df_main["统一社会信用代码"] == "91441300XXXXXXXXXX"
df_main.loc[mask, "数据状态"] = "作废"
# 查询时过滤作废数据：df_main[df_main["数据状态"]=="正常"]

In [ ]:
# ========== 实战演示 ==========
manager = EnterpriseManager('企业数据_清洗版.xlsx')

# 1. 查询：找制造业且注册资本>=500万、非失信的企业
results = manager.search(
    industry='制造业',
    capital_gte=500,
    is_dishonest=False
)
print(f"找到 {len(results)} 家符合条件的企业")

In [ ]:
# 2. 添加新企业
manager.add_company(
    company='XX科技有限公司',
    credit_code='91110000MA01XXXXX',
    capital=1000,
    status='存续',
    legal='张三',
    phone='13800138000',
    industry='科技推广'
)

In [ ]:
# 3. 更新：修改某企业的联系电话
manager.update(
    credit_code='91110000MA01XXXXX',
    phone='13900139000',
    is_trusted=True
)

In [ ]:
# 4. 删除：删除某家已注销企业
manager.delete(credit_code='91110000MA01YYYYY')

In [ ]:
# 5. 保存
manager.save()